### Build a RAG pipeline

### This Pipeline  - Data Ingestion -> Vector DB 

In [21]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\G_AI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
class EmbeddingManager:

    def __init__(self, model_name : str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding model

        Args:
            model_name(str): Hugging face model name for sentence embeddings
        """
        self.model_name = model_name
        self.model  = None
        self._load_model() # Automatically trigger the loading when we call the class actually

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            dimension = self.model.get_sentence_embedding_dimension()
            print(f"Model loaded successfully. Embedding dimension: {dimension}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## Initialize the embeddiing manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


d:\G_AI\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\teraa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3013.57it/s]
BertModel LOAD REPORT from: sentence-transf

Model loaded successfully. Embedding dimension: 384


### Vectorstore

In [30]:
import chromadb
from chromadb.config import Settings

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
    
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
    
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        """
        # 1. Validation Check
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # 2. Preparation Lists
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        # 3. Data Processing Loop
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Prepare Content
            documents_text.append(doc.page_content)

            # Prepare Embedding (Convert NumPy to List)
            embeddings_list.append(embedding.tolist())

        # 4. Storage (The "Commit")
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
### Read all the PDF's inside the directory

from pathlib import Path
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader,
    UnstructuredPowerPointLoader,
    TextLoader
)

def process_all_files(directory_path):
    """
    A generic loader that handles PDF, PPTX, and TXT files 
    recursively from a parent directory.
    """
    parent_dir = Path(directory_path)
    
    # Mapping extensions to their respective LangChain Loaders
    loaders = {
        ".pdf": PyPDFLoader,
        ".pptx": UnstructuredPowerPointLoader,
        ".txt": TextLoader
    }
    
    all_documents = []

    # Iterate through the defined extensions
    for ext, loader_class in loaders.items():
        print(f"--- Processing {ext} files ---")
        
        # DirectoryLoader handles the heavy lifting of finding files and loading them
        loader = DirectoryLoader(
            path=str(parent_dir),
            glob=f"**/*{ext}",
            loader_cls=loader_class,
            show_progress=False,
            use_multithreading=True # Speeds up processing for many files
        )
        
        try:
            docs = loader.load()
            
            # Enrich metadata just like your previous function
            for doc in docs:
                file_path = Path(doc.metadata.get('source', ''))
                doc.metadata['source_file'] = file_path.name
                doc.metadata['file_type'] = ext.replace('.', '')
            
            all_documents.extend(docs)
            print(f" ✓ Successfully loaded {len(docs)} segments from {ext} files\n")
            
        except Exception as e:
            print(f" X Error loading {ext} files: {e}")

    print(f"Final Count: {len(all_documents)} total document segments loaded.")
    return all_documents

# Execution
all_data = process_all_files("../data")

--- Processing .pdf files ---
 ✓ Successfully loaded 370 segments from .pdf files

--- Processing .pptx files ---
 ✓ Successfully loaded 0 segments from .pptx files

--- Processing .txt files ---
 ✓ Successfully loaded 3 segments from .txt files

Final Count: 373 total document segments loaded.


In [16]:
all_data

[Document(metadata={'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Basic Process Control.docx', 'source': '..\\data\\pdf\\Basic Process Control.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'Basic Process Control.pdf', 'file_type': 'pdf'}, page_content='Basic  Process  Control   \nObjective:  To  learn  the  basic  process  management  commands  in  a  Linux  machine.   Tools:  Virtual  Machine,  Linux  environment  (CentOS)   Prerequisites  :  No  Prerequisites   Task:  Perform  various  commands  on  process  management  practically.   1.  ps  aux  :  This  command  displays  information  about  all  running  processes  on  the  \nsystem,\n \nincluding\n \nthose\n \nnot\n \nassociated\n \nwith\n \na\n \nterminal.\n \nThe\n \noptions\n \nare:\n \n●  a :  Show  processes  for  all  users.  ●  u :  Display  user-oriented  output  (includes  usernames  and  resource  usage).  ●  x :  Include  processes  without 

In [19]:
## text Splitting into chunks

def split_documents(documents,chunk_size=500,chunk_overlap=50):
    """
    This functions Splits the document text into chunks for better RAG performance

    Args:
        documents(List) : Pass the list of documents.
        chunk_size(int) :  Split all the documents based on the chunk_size.
        chuck_overlap(int) : How much size one document to another document chunk can happen.
    Return:

    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n",'\n'," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [20]:
chunks = split_documents(all_data)
chunks

Split 373 documents into 727 chunks

Example chunk:
Content: Basic  Process  Control   
Objective:  To  learn  the  basic  process  management  commands  in  a  Linux  machine.   Tools:  Virtual  Machine,  Linux  environment  (CentOS)   Prerequisites  :  No  Pr...
Metadata: {'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Basic Process Control.docx', 'source': '..\\data\\pdf\\Basic Process Control.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'Basic Process Control.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Basic Process Control.docx', 'source': '..\\data\\pdf\\Basic Process Control.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'Basic Process Control.pdf', 'file_type': 'pdf'}, page_content='Basic  Process  Control   \nObjective:  To  learn  the  basic  process  management  commands  in  a  Linux  machine.   Tools:  Virtual  Machine,  Linux  environment  (CentOS)   Prerequisites  :  No  Prerequisites   Task:  Perform  various  commands  on  process  management  practically.   1.  ps  aux  :  This  command  displays  information  about  all  running  processes  on  the  \nsystem,\n \nincluding\n \nthose\n \nnot\n \nassociated\n \nwith\n \na\n \nterminal.\n \nThe\n \noptions\n \nare:'),
 Document(metadata={'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Basic Process Control.docx', 'source': '..\\da

### Embedding and Vector Store DB

In [31]:
### Covert text to embeddings

texts = [doc.page_content for doc in chunks]


### Generate the Embedding

embeddings = embedding_manager.generate_embeddings(texts)

### Store in the vector Database
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 727 texts...


Batches: 100%|██████████| 23/23 [00:15<00:00,  1.53it/s]


Generated embeddings with shape: (727, 384)
Adding 727 documents to vector store...
Successfully added 727 documents to vector store
Total documents in collection: 727


### Retrival Pipeline from the vector Database

In [32]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store, embedding_manager):
        """
        Initialize the retriever
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [33]:
rag_retriever

In [34]:
rag_retriever.retrieve("Explain about the ingress concepts from kubernetes")

Retrieving documents for query: 'Explain about the ingress concepts from kubernetes'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.91it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_a90a1ecc_429',
  'content': 'Kubernetes Ingress \n In the world of Kubernetes, Ingress is your ticket to managing external traffic to services \n within the cluster. Before we dive into the details, let’s recap what we’ve learned so far. \n Before Ingress, the Service provides a Load balancer, which is used to distribute the traffic \n between multiple applications or pods. \n Ingress helps to expose the HTTP and HTTPS routes from outside of the cluster. \n Ingress enables Path-based and Host-based routing.',
  'metadata': {'creator': 'PyPDF',
   'total_pages': 293,
   'page_label': '137',
   'content_length': 477,
   'producer': 'PyPDF',
   'creationdate': '',
   'page': 136,
   'source_file': 'Kubernetes Complete Material.pdf',
   'doc_index': 429,
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\Kubernetes Complete Material.pdf'},
  'similarity_score': 0.6608692705631256,
  'distance': 0.3391307294368744,
  'rank': 1},
 {'id': 'doc_c8c942d2_430',
  'content': 'Ingres

In [37]:
rag_retriever.retrieve("Explain about the EKS,AKS,GKE")

Retrieving documents for query: 'Explain about the EKS,AKS,GKE'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.62it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_32752dcb_588',
  'content': 'Azure Kubernetes Service(AKS) \n What is AKS? \n AKS is a managed Kubernetes Service that helps us to deploy our applications without \n worrying about Control Plane and other things like regular updates support, Scaling, and \n high availability of your cluster. \n Key Features: \n ●  Managed Control Plane  : AKS provides a fully managed  control plane. So that, you \n don’t need to configure things like the upgradation of Kubernetes cluster, patching,',
  'metadata': {'page_label': '205',
   'file_type': 'pdf',
   'page': 204,
   'source_file': 'Kubernetes Complete Material.pdf',
   'source': '..\\data\\pdf\\Kubernetes Complete Material.pdf',
   'doc_index': 588,
   'total_pages': 293,
   'producer': 'PyPDF',
   'content_length': 450,
   'creator': 'PyPDF',
   'creationdate': ''},
  'similarity_score': 0.19244498014450073,
  'distance': 0.8075550198554993,
  'rank': 1},
 {'id': 'doc_336966d4_565',
  'content': 'AWS Elastic Kubernetes \n Servi

### Integration vector DB context pipeline with LLM output

### RAG pipeline with GROQ LLM

In [50]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the GROQ llm (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0.1,
    max_tokens=1024
)

## Create a Simple RAG function: retrive context + generate response
def rag_simple(query,retriver,llm,top_k=3):
    ## retrive the context 
    results = retriver.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return " No relevant context found to answer the question"
    ## generate the answer using the groq LLM
    prompt = f"""Use the following context to answer the question concisely.
            If the answer is not in the context, say you don't know.

            Context:
            {context}

            Question: {query}

            Answer:"""

    print("--- GENERATED PROMPT ---")
    print(prompt)
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [ ]:
answer =  rag_simple("Explain deeply about EKS,AKS, GKE?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'How to deploy application and app should be accessible to the public'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.74it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
 No relevant context found to answer the question


In [55]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Explain deeply about the interview question on kubernetes clusters ?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Explain deeply about the interview question on kubernetes clusters ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Here's a concise answer to an interview question about Kubernetes clusters:

**What is a Kubernetes Cluster?**

A Kubernetes cluster is a set of nodes that work together to run containerized applications. It consists of at least one Master Node, which controls the cluster, and multiple Worker Nodes, which manage and run the containers.

**Components of a Kubernetes Cluster:**

1. **Master Node**: The Master Node is the control plane of the cluster, responsible for managing the cluster's configuration, scheduling applications, and maintaining the desired state.
2. **Worker Nodes**: Worker Nodes are the machines that run the containers. They communicate with the Master Node to receive instructions and report the status of the containers.

**Key Features of a Kubernetes Cluster:**

1. **Scalability**: Kubernetes clusters can scale horizontally by adding more Worker Nodes to handle increased traffic or workload.
2. **High Availability**: Kubernetes clusters ensure high availability

In [56]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Explain how we externalize the applications using kubernetes so that users can access the applications", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Explain how we externalize the applications using kubernetes so that users can access the applications'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.85it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Kubernetes Ingress 
 In the world of Kubernetes, Ingress is your ticket to managing external traffic to services 
 within the cluster. Before we dive into the details, let’s recap what we’ve learned so far. 
 Before Ingress, the Service provides a Loa

d balancer, which is used to distribute the traffic 
 between multiple applications or pods. 
 Ingress helps to expose the HTTP and HTTPS routes from outside of the cluster. 
 Ingress enables Path-based and Host-based routing.

Kubernetes Networking (Services) 
 Objectives 
 By the end of this topic, you will: 
 Understand the basics of how pods and containers can communicate within the same pod 
 and node. 
 Explore the critical role of Service objects in Kubernetes networking. 
 Gain insights into different Service types, including ClusterIP, NodePort, LoadBalancer, and 
 ExternalName. 
 Things to know about accessing pods or containers in some scenarios:

●  Multiple containers within the pod access each other through a loopback 
 address(localhost). 
 ●  The cluster provides communication between multiple pods. 
 ●  To access your application from outside of the cluster, you need a Services object in 
 Kubernetes. 
 ●  You can also use the Services object to publish services only f